# 09 · Taxi and licence checks: silver, and the taxi_ and lic_ features

- **Taxi:** a trip is known once it ends, so silver counts trips by the hour they ended, per pickup area. Trips without a pickup area (7% in the EDA) count only citywide.
- **Licences:** a venue (site) counts as active at T if one of its licences was issued before T, had started, and hadn't expired. Only the licence types in config's two groups are kept: nightlife and package goods.

This notebook checks silver against bronze, the features for June 2019, a first look at their relation to crashes, and one value of each recomputed by hand.

In [1]:
import os
from datetime import datetime, timedelta

import pandas as pd
import pyspark.sql.functions as F
import yaml

# run from the repo root, where main.py, config/ and data/ are
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import main
from utils.gold import target_hours
from utils.licences import lic_features
from utils.taxi import taxi_features

with open("config/pipeline.yaml") as f:
    config = yaml.safe_load(f)
spark = main.get_spark(config)

bronze_taxi = spark.read.parquet("datamart/bronze/taxi")
silver_taxi = spark.read.parquet("datamart/silver/taxi")
bronze_licences = spark.read.parquet("datamart/bronze/licences")
silver_licences = spark.read.parquet("datamart/silver/licences")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/17 11:15:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1 · Taxi: silver against bronze

In [2]:
trips = bronze_taxi.count()
by_area = silver_taxi.groupBy("community_area").agg(F.sum("trips").alias("trips")).toPandas()
no_area = by_area.loc[by_area["community_area"].isna(), "trips"].sum()
print(f"bronze trips:           {trips:,}")
print(f"silver trips:           {by_area['trips'].sum():,} in {silver_taxi.count():,} area-hours")
print(f"  without pickup area:  {no_area:,} ({no_area / trips:.1%})")
share = by_area.dropna().set_index("community_area")["trips"].sort_values(ascending=False) / by_area["trips"].dropna().sum()
print(f"top 4 areas' share of trips with an area: {share.head(4).sum():.1%} ({', '.join(str(int(a)) for a in share.index[:4])})")

bronze trips:           48,176,932
silver trips:           48,176,932 in 1,780,500 area-hours
  without pickup area:  3,481,404 (7.2%)
top 4 areas' share of trips with an area: 71.4% (8, 32, 76, 28)


In [3]:
monthly = (silver_taxi.groupBy(F.date_format("hour_start", "yyyy-MM").alias("month")).agg(F.sum("trips").alias("trips"))
           .toPandas().set_index("month")["trips"].sort_index())
monthly.loc[["2019-04", "2020-04", "2021-06", "2022-12"]]

month
2019-04    1458186
2020-04      55585
2021-06     332137
2022-12     513512
Name: trips, dtype: int64

## 2 · Licences: what silver keeps

In [4]:
groups = config["features"]["licence_groups"]
print(f"bronze licences: {bronze_licences.count():,}; in silver: {silver_licences.count():,}")
(silver_licences.groupBy("licence_group")
 .agg(F.count("*").alias("licences"), F.countDistinct("account_number", "site_number").alias("sites"),
      F.countDistinct("community_area").alias("areas"))
 .toPandas().set_index("licence_group"))

bronze licences: 27,305; in silver: 17,478


,licences,sites,areas
licence_group,,,
package_goods,4041,1418,76
nightlife,13437,4511,75


## 3 · The features for June 2019

In [5]:
targets = target_hours(datetime(2019, 6, 1), datetime(2019, 7, 1), config)
taxi = taxi_features(targets, config, spark).toPandas()
lic = lic_features(targets, config, spark).toPandas()
features = taxi.merge(lic, on=["community_area", "hour_start"])
print(f"{len(features):,} rows = 77 areas x {len(targets)} hours; missing values: {features.isna().sum().sum()}")
print("area-hours with no trip ended:", f"{(features['taxi_area_trips_1h'] == 0).mean():.1%}")
print("areas with no active nightlife venue on 1 June:",
      (features.loc[features["hour_start"] == targets[0], "lic_nightlife_active"] == 0).sum())
features.drop(columns=["community_area", "hour_start"]).describe().round(1)

55,440 rows = 77 areas x 720 hours; missing values: 0
area-hours with no trip ended: 45.5%
areas with no active nightlife venue on 1 June: 2


,taxi_area_trips_1h,taxi_city_trips_1h,lic_nightlife_active,lic_package_goods_active
count,55440.0,55440.0,55440.0,55440.0
mean,25.0,2071.3,44.0,14.7
std,117.2,1164.5,84.2,16.1
min,0.0,163.0,0.0,0.0
25%,0.0,1055.8,6.0,5.0
50%,1.0,2098.5,13.0,8.0
75%,4.0,3069.2,40.0,18.0
max,1582.0,4640.0,536.0,79.0


## 4 · Do they relate to crashes?

June 2019's labels against each feature, in bands. A first look, not a model: busy areas have more of everything.

In [6]:
labels = pd.read_parquet("datamart/gold/label_store", filters=[("date", ">=", "2019-06-01"), ("date", "<=", "2019-07-01")])
labels = labels.sort_values("label_snapshot_date").drop_duplicates(["community_area", "hour_start"], keep="last")
joined = features.merge(labels[["community_area", "hour_start", "crash_count"]], on=["community_area", "hour_start"])
taxi_bands = pd.cut(joined["taxi_area_trips_1h"], [0, 1, 10, 100, 10_000], right=False)
nightlife_bands = pd.cut(joined["lic_nightlife_active"], [0, 1, 20, 100, 1_000], right=False)
pd.concat({"taxi_area_trips_1h": joined.groupby(taxi_bands, observed=True)["crash_count"].agg(["mean", "count"]),
           "lic_nightlife_active": joined.groupby(nightlife_bands, observed=True)["crash_count"].agg(["mean", "count"])}).round(3)

mean  count
taxi_area_trips_1h   [0, 1)        0.116  25227
                     [1, 10)       0.195  21835
                     [10, 100)     0.297   5878
                     [100, 10000)  0.651   2500
lic_nightlife_active [0, 1)        0.067   1440
                     [1, 20)       0.142  32450
                     [20, 100)     0.193  16510
                     [100, 1000)   0.533   5040

## 5 · Point in time: one value of each by hand

At 18:00 on Friday 14 June 2019 in the Loop (area 32): taxi trips that ended 17:00–18:00, counted straight from bronze; and active nightlife sites, from bronze licences with pandas.

In [7]:
T, area = datetime(2019, 6, 14, 18), 32
ended = bronze_taxi.filter((F.col("_arrived_at") >= T - timedelta(hours=1)) & (F.col("_arrived_at") < T))
trips_by_hand = ended.filter(F.col("pickup_community_area").cast("int") == area).count()

lic = bronze_licences.filter(F.col("license_description").isin(groups["nightlife"])
                             & (F.col("community_area").cast("int") == area)).toPandas()
start, expires = pd.to_datetime(lic["license_start_date"]), pd.to_datetime(lic["expiration_date"])
active = lic[(lic["_arrived_at"] < T) & (start <= T) & (expires.isna() | (expires > T))]
sites_by_hand = active[["account_number", "site_number"]].drop_duplicates().shape[0]

row = features[(features["community_area"] == area) & (features["hour_start"] == T)].iloc[0]
print(f"taxi trips ended 17:00-18:00: by hand {trips_by_hand}, taxi_features {row['taxi_area_trips_1h']:.0f}")
print(f"active nightlife sites:       by hand {sites_by_hand}, lic_features {row['lic_nightlife_active']:.0f}")

taxi trips ended 17:00-18:00: by hand 1269, taxi_features 1269
active nightlife sites:       by hand 282, lic_features 282
